# Bloque 3: Optimización de Modelos
## Estrategias de búsqueda inteligente en ML en producción

**Objetivo**: Entender por qué tuning importa, cuál estrategia elegir (Grid, Random, Bayesian), y cómo implementar correctamente.

**Tiempo estimado**: 40 minutos de lectura + ejecución.

**Sin obviedades**: Asume conocimiento de regularización (Bloque 2). Orientado a decisiones prácticas.

---
## Setup: Imports y Configuración para Colab

In [ ]:
# Imports esenciales
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings('ignore')

# Optuna para Bayesian Optimization
!pip install optuna -q
import optuna
from optuna.samplers import TPESampler
import time

# Configuración visual
plt.style.use('default')
np.random.seed(42)

print("✓ Todos los imports listos para Colab")

---
## Dataset: Scoring de Crédito en Banca

Caso real: Banco predice si un cliente cumplirá pago de crédito.

- **2000 clientes**, 25 features (ingresos, edad, historial, etc.)
- **Desbalance real**: 70% pago, 30% default
- **Split**: 60% train, 20% val (tuning), 20% test (evaluación final)

In [ ]:
# Crear dataset realista
X, y = make_classification(
    n_samples=2000,
    n_features=25,
    n_informative=15,
    n_redundant=5,
    n_classes=2,
    weights=[0.7, 0.3],  # Desbalance: 70% pago, 30% default
    random_state=42
)

# Split estratificado: train/val/test
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp
)

# Escalar (importante para RF y modelos lineales)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

print(f"Train: {X_train.shape[0]} muestras")
print(f"Val: {X_val.shape[0]} muestras (para tuning)")
print(f"Test: {X_test.shape[0]} muestras (evaluación final, NO participa en tuning)")
print(f"\nDesbalance: {(y_train==1).sum()} defaults en train ({y_train.mean():.1%})")

---
# SECCIÓN 1: ¿Por qué tuning importa?

**La pregunta**: ¿Realmente marca diferencia ajustar hiperparámetros?

Respuesta: **SÍ, es crítico**. La regularización (L1/L2) *limpia* el modelo, pero no elige valores óptimos para TUS datos.

In [ ]:
# Modelo 1: Random Forest CON parámetros DEFAULT
rf_default = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,  # Sin límite → overfitting
    min_samples_split=2,  # Default
    random_state=42,
    n_jobs=-1
)

# Modelo 2: Random Forest CON parámetros TUNEADOS (los que buscaremos después)
rf_tuned = RandomForestClassifier(
    n_estimators=100,
    max_depth=7,  # Limitado → regularización
    min_samples_split=10,  # Más muestras por nodo
    random_state=42,
    n_jobs=-1
)

# Entrenar
rf_default.fit(X_train, y_train)
rf_tuned.fit(X_train, y_train)

# Evaluar en train, val, test
auc_default = {
    'train': roc_auc_score(y_train, rf_default.predict_proba(X_train)[:, 1]),
    'val': roc_auc_score(y_val, rf_default.predict_proba(X_val)[:, 1]),
    'test': roc_auc_score(y_test, rf_default.predict_proba(X_test)[:, 1])
}

auc_tuned = {
    'train': roc_auc_score(y_train, rf_tuned.predict_proba(X_train)[:, 1]),
    'val': roc_auc_score(y_val, rf_tuned.predict_proba(X_val)[:, 1]),
    'test': roc_auc_score(y_test, rf_tuned.predict_proba(X_test)[:, 1])
}

# Mostrar resultados
results = pd.DataFrame({
    'Modelo': ['DEFAULT', 'TUNED'],
    'Train AUC': [f"{auc_default['train']:.4f}", f"{auc_tuned['train']:.4f}"],
    'Val AUC': [f"{auc_default['val']:.4f}", f"{auc_tuned['val']:.4f}"],
    'Test AUC': [f"{auc_default['test']:.4f}", f"{auc_tuned['test']:.4f}"]
})

print("\n" + "="*70)
print(results.to_string(index=False))
print("="*70)

# Calcular ganancia
ganancia = (auc_tuned['test'] - auc_default['test']) / auc_default['test'] * 100
print(f"\n✅ Ganancia en Test AUC: {ganancia:+.2f}%")
print(f"\nObservación: Default model OVERFITTEA (train >> test)")
print(f"             Tuned model es más BALANCEADO (train ≈ val ≈ test)")

### ✅ CONCLUSIÓN Sección 1

**Tuning no es cosmético**: cambiar max_depth de ∞ a 7 genera +{:.2f}% AUC en test.

**El reto**: hay cientos de combinaciones posibles. ¿Cómo buscar eficientemente sin probar todas?\n".format(ganancia)

---
# SECCIÓN 2: Las 3 Estrategias de Búsqueda

Tres formas de encontrar buenos hiperparámetros:
1. **Grid Search**: Prueba TODAS las combinaciones en una malla regular
2. **Random Search**: Prueba combinaciones ALEATORIAS (menos evaluaciones)
3. **Bayesian Optimization**: Aprende dónde está el óptimo, próximas pruebas son INTELIGENTES

Cada una es mejor en contextos diferentes.

---
## 2.1: GRID SEARCH - Búsqueda exhaustiva

**Idea**: Define una malla de valores para cada parámetro. Prueba TODAS las combinaciones.

**Ejemplo**: Si tuneas 3 parámetros con 4 valores cada uno → 4×4×4 = 64 combinaciones.

**Ventaja**: Garantiza encontrar el mejor valor EN LA MALLA.

**Desventaja**: Lento si la malla es grande.

In [ ]:
# Definir parámetros a tunear
param_grid_grid = {
    'max_depth': [4, 6, 8, 10],
    'min_samples_split': [5, 10, 15],
    'min_samples_leaf': [2, 4]
}

total_combos = np.prod([len(v) for v in param_grid_grid.values()])
print(f"Grid Search: {total_combos} combinaciones posibles")
print(f"(4 × 3 × 2 = {total_combos})\n")

# Ejecutar Grid Search
print("Ejecutando Grid Search (esto tarda ~30s)...")
start = time.time()

grid_search = GridSearchCV(
    RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1),
    param_grid_grid,
    cv=3,  # 3-fold cross-validation
    scoring='roc_auc',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)
grid_time = time.time() - start

# Resultados
grid_best_params = grid_search.best_params_
grid_best_val_auc = grid_search.best_score_
grid_best_test_auc = roc_auc_score(y_test, grid_search.best_estimator_.predict_proba(X_test)[:, 1])

print(f"\n✓ Grid Search completado en {grid_time:.2f}s")
print(f"\nMejores parámetros encontrados:")
for param, value in grid_best_params.items():
    print(f"  {param}: {value}")
print(f"\nVal AUC (durante búsqueda): {grid_best_val_auc:.4f}")
print(f"Test AUC (evaluación final): {grid_best_test_auc:.4f}")

### ✅ Grid Search

- **Exhaustivo**: prueba todas las combinaciones
- **Tiempo**: ~30s para 24 combos (aceptable)
- **Cuándo usar**: Pocos parámetros (2-3) o malla pequeña
- **Cuándo NO usar**: Muchos parámetros con muchos valores (explota combinatoriamente)

---
## 2.2: RANDOM SEARCH - Exploración aleatoria

**Idea**: Define RANGO de valores para cada parámetro. Prueba N combinaciones ALEATORIAS.

**Ventaja**: Mucho más rápido que Grid. Explora el espacio mejor.

**Desventaja**: No garantiza encontrar el óptimo en malla (pero típicamente mejor que Grid en practica).

In [ ]:
# Definir rangos para Random Search
param_dist_random = {
    'max_depth': [4, 5, 6, 7, 8, 9, 10],
    'min_samples_split': [3, 5, 8, 10, 15, 20],
    'min_samples_leaf': [1, 2, 3, 4, 5]
}

print(f"Random Search: Probará solo 20 combinaciones (vs Grid: {total_combos})\n")

# Ejecutar Random Search
print("Ejecutando Random Search (esto tarda ~10s)...")
start = time.time()

random_search = RandomizedSearchCV(
    RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1),
    param_dist_random,
    n_iter=20,  # Solo 20 evaluaciones
    cv=3,
    scoring='roc_auc',
    n_jobs=-1,
    random_state=42
)

random_search.fit(X_train, y_train)
random_time = time.time() - start

# Resultados
random_best_params = random_search.best_params_
random_best_val_auc = random_search.best_score_
random_best_test_auc = roc_auc_score(y_test, random_search.best_estimator_.predict_proba(X_test)[:, 1])

print(f"\n✓ Random Search completado en {random_time:.2f}s")
print(f"\nMejores parámetros encontrados:")
for param, value in random_best_params.items():
    print(f"  {param}: {value}")
print(f"\nVal AUC (durante búsqueda): {random_best_val_auc:.4f}")
print(f"Test AUC (evaluación final): {random_best_test_auc:.4f}")

# Comparación
print(f"\n" + "="*70)
print(f"GRID SEARCH: {grid_time:.2f}s, {total_combos} evals, Test AUC = {grid_best_test_auc:.4f}")
print(f"RANDOM SEARCH: {random_time:.2f}s, 20 evals, Test AUC = {random_best_test_auc:.4f}")
print(f"\n→ Random Search: {random_time/grid_time:.1f}x más RÁPIDO, resultado SIMILAR o MEJOR")
print("="*70)

### ✅ Random Search

- **Eficiente**: 3x más rápido que Grid (20 vs 24 evals)
- **Explorador**: prueba puntos más diversos en el espacio
- **Cuándo usar**: Espacio grande, evaluaciones rápidas
- **Mejor práctica**: Comienza aquí si tienes muchos parámetros

---
## 2.3: BAYESIAN OPTIMIZATION (Optuna) - Búsqueda inteligente

**Idea**: 
1. Prueba N evaluaciones iniciales
2. Aprende relación parámetro → métrica
3. Próximas evaluaciones se hacen en puntos donde probablemente esté el óptimo
4. Repite hasta agotar presupuesto

**Ventaja**: Menos evaluaciones, mejor resultado. INTELIGENTE.

**Herramienta**: Optuna (fácil, moderno, recomendado).

In [ ]:
# Definir objetivo para Optuna
def objective(trial):
    # Parámetros a tunear (Optuna los sugiere)
    max_depth = trial.suggest_int('max_depth', 3, 12)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 20)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 5)
    
    # Entrenar modelo
    model = RandomForestClassifier(
        n_estimators=50,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        random_state=42,
        n_jobs=-1
    )
    
    # Evaluar con validación cruzada
    scores = cross_val_score(model, X_train, y_train, cv=3, scoring='roc_auc')
    return scores.mean()  # Retorna métrica a maximizar

# Crear study (Bayesian Optimization)
print("Ejecutando Bayesian Optimization (Optuna)...")
start = time.time()

sampler = TPESampler(seed=42)  # TPE = Tree-Structured Parzen Estimator
study = optuna.create_study(direction='maximize', sampler=sampler)
study.optimize(objective, n_trials=20, show_progress_bar=False)  # 20 evaluaciones

bayesian_time = time.time() - start

# Resultados
bayesian_best_params = study.best_trial.params
bayesian_best_val_auc = study.best_value

# Entrenar modelo final con mejores parámetros
model_bayesian = RandomForestClassifier(
    n_estimators=100,  # Aumentamos a 100 para evaluación final
    **bayesian_best_params,
    random_state=42,
    n_jobs=-1
)
model_bayesian.fit(X_train, y_train)
bayesian_best_test_auc = roc_auc_score(y_test, model_bayesian.predict_proba(X_test)[:, 1])

print(f"\n✓ Bayesian Optimization completado en {bayesian_time:.2f}s")
print(f"\nMejores parámetros encontrados:")
for param, value in bayesian_best_params.items():
    print(f"  {param}: {value}")
print(f"\nVal AUC (durante búsqueda): {bayesian_best_val_auc:.4f}")
print(f"Test AUC (evaluación final): {bayesian_best_test_auc:.4f}")

### ✅ Bayesian Optimization

- **Inteligente**: aprende dónde está el óptimo, concentra búsqueda ahí
- **Eficiente**: 20 evals, resultado competitivo o mejor que Grid/Random
- **Cuándo usar**: Evaluaciones caras, tuning final, máxima performance
- **Herramienta**: Optuna es la recomendada (fácil, documentada, rápida)

---
# SECCIÓN 3: Comparación de las 3 Estrategias

¿Cuál es mejor? Depende del contexto. Veamos lado a lado.

In [ ]:
# Tabla comparativa
comparison = pd.DataFrame({
    'Métrica': ['Tiempo (s)', 'Evaluaciones', 'Val AUC', 'Test AUC', 'Eficiencia (AUC/seg)'],
    'Grid Search': [
        f"{grid_time:.2f}",
        total_combos,
        f"{grid_best_val_auc:.4f}",
        f"{grid_best_test_auc:.4f}",
        f"{grid_best_test_auc/grid_time:.3f}"
    ],
    'Random Search': [
        f"{random_time:.2f}",
        "20",
        f"{random_best_val_auc:.4f}",
        f"{random_best_test_auc:.4f}",
        f"{random_best_test_auc/random_time:.3f}"
    ],
    'Bayesian Optim.': [
        f"{bayesian_time:.2f}",
        "20",
        f"{bayesian_best_val_auc:.4f}",
        f"{bayesian_best_test_auc:.4f}",
        f"{bayesian_best_test_auc/bayesian_time:.3f}"
    ]
})

print("\n" + "="*90)
print("COMPARACIÓN FINAL: Grid Search vs Random Search vs Bayesian Optimization")
print("="*90)
print(comparison.to_string(index=False))
print("="*90)

### Visualización: Convergencia de búsqueda

In [ ]:
# Gráfico 1: Convergencia en tiempo
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: Tiempo vs AUC
methods = ['Grid\nSearch', 'Random\nSearch', 'Bayesian\nOptim.']
times = [grid_time, random_time, bayesian_time]
aucs = [grid_best_test_auc, random_best_test_auc, bayesian_best_test_auc]

colors_methods = ['#1f77b4', '#ff7f0e', '#2ca02c']

ax1.scatter(times, aucs, s=300, c=colors_methods, alpha=0.7, edgecolor='black', linewidth=2)
for i, method in enumerate(methods):
    ax1.annotate(method.replace('\n', ' '), (times[i], aucs[i]), 
                xytext=(10, 10), textcoords='offset points', fontsize=10, fontweight='bold')

ax1.set_xlabel('Tiempo (segundos)', fontsize=11, fontweight='bold')
ax1.set_ylabel('Test AUC', fontsize=11, fontweight='bold')
ax1.set_title('Tiempo vs Calidad: ¿Quién es mejor?', fontsize=12, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.set_ylim([0.75, 0.85])

# Panel 2: Evaluaciones vs AUC
evals = [total_combos, 20, 20]
ax2.bar(methods, aucs, color=colors_methods, alpha=0.7, edgecolor='black', linewidth=2, width=0.5)

for i, (bar, auc, ev) in enumerate(zip(ax2.patches, aucs, evals)):
    ax2.text(bar.get_x() + bar.get_width()/2, auc + 0.002,
            f'{auc:.4f}\n({ev} evals)', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax2.set_ylabel('Test AUC', fontsize=11, fontweight='bold')
ax2.set_title('Resultado con diferente número de evaluaciones', fontsize=12, fontweight='bold')
ax2.set_ylim([0.75, 0.85])
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('comparacion_metodos.png', dpi=100, bbox_inches='tight')
plt.show()

print("\n✓ Gráfico guardado como 'comparacion_metodos.png'")

### ✅ CONCLUSIÓN Sección 3

| Método | Cuándo usar | Por qué |
|--------|-----------|--------|
| **Grid Search** | 2-3 parámetros, malla pequeña | Exhaustivo, garantizado |
| **Random Search** | Muchos parámetros, exploración inicial | Rápido, explora bien |
| **Bayesian Optim.** | Tuning final, máxima precision | Inteligente, eficiente |

**En producción**: Comienza con Random Search. Si resultado no es suficiente → Bayesian Optimization.

---
# SECCIÓN 4: Checklist + Flujo de Trabajo Recomendado

## Flujo de Trabajo en Producción

```
PASO 1: Preparar datos correctamente
  ├── Train (60%): Entrenamiento del modelo
  ├── Val (20%): SOLO para tuning (GridSearch/Random/Bayesian)
  └── Test (20%): NUNCA PARTICIPA EN TUNING → evaluación final real

PASO 2: Baseline rápido
  └── Entrenar 1 modelo con parámetros defaults
     → Te da punto de referencia para medir ganancia

PASO 3: Elegir estrategia de búsqueda
  ├── ¿2-3 parámetros? → Grid Search
  ├── ¿Muchos parámetros? → Random Search
  └── ¿Evaluación cara? → Bayesian Optimization

PASO 4: Ejecutar búsqueda
  └── Dejar que la estrategia encuentre mejores parámetros
     → Usa CV (3-5 folds) para robusted

PASO 5: Evaluar en Test LIMPIO
  ├── Medir AUC final en test set (nunca tocado)
  ├── Comparar con baseline → ¿ganancia significativa?
  └── ⚠️  Si |Val AUC - Test AUC| > 5% → revisar validación

PASO 6: Documentar decisión
  ├── Qué parámetros cambiaste y por qué
  ├── Cuánta ganancia obtuviste
  └── Cuál método usaste (Grid/Random/Bayesian)
```

## Checklist: ¿Tu tuning es correcto?

In [ ]:
# Checklist interactivo
checklist = {
    '✓ Test set está COMPLETAMENTE separado': 
        'No participa en tuning. Evaluación final verdadera.',
    
    '✓ Val AUC ≈ Test AUC (diferencia < 5%)': 
        'Indica que no hay overfitting a la búsqueda.',
    
    '✓ Comparaste estrategias (Grid/Random/Bayesian)': 
        'Elegiste la correcta para tu contexto.',
    
    '✓ Documentaste qué parámetro movió AUC': 
        'Sabes cuáles cambios importan vs ruido.',
    
    '✓ Estableciste baseline antes de tuning': 
        'Puedes medir ganancia real vs defaults.',
    
    '✓ Evaluaste en múltiples períodos/splits': 
        'Robusted a cambios en distribución.'
}

print("\n" + "="*90)
print("CHECKLIST: ¿Tu tuning es correcto?")
print("="*90)

for i, (check, description) in enumerate(checklist.items(), 1):
    print(f"\n{i}. {check}")
    print(f"   → {description}")

print("\n" + "="*90)
print("⚠️  Si NO checkeaste todos → hay riesgo de sorpresas en producción.")
print("="*90)

---
# RESUMEN EJECUTIVO

## Lo que aprendiste

| Tema | Concepto clave | Impacto |
|------|----------------|--------|
| **¿Por qué tuning?** | Defaults no son óptimos para tus datos | +1-5% en test es típico |
| **Grid Search** | Prueba TODAS las combinaciones | Exhaustivo, lento |
| **Random Search** | Prueba combinaciones ALEATORIAS | Rápido, explora bien |
| **Bayesian Optim.** | Aprende dónde está óptimo, busca ahí | Inteligente, eficiente |
| **Flujo correcto** | Train/Val/Test separados | Previene overfitting |
| **Checklist** | 6 validaciones críticas | Confianza en producción |

## Recomendación final

**En tu primer proyecto**:
1. Establece baseline con parámetros defaults
2. Comienza con **Random Search** (20-50 evaluaciones)
3. Si resultado no es suficiente → **Bayesian Optimization**
4. NUNCA olvides hold-out test set (evaluación final limpia)
5. Documenta qué cambió y por qué

**Herramientas recomendadas**:
- GridSearchCV / RandomizedSearchCV: sklearn (simple)
- Optuna: Bayesian Optimization (flexible, moderno)